In [1]:
# ============================================================
# Cell 1: Spin-1 operators + MPO construction
# ============================================================
import numpy as np
import os
os.makedirs("figureAKLT", exist_ok=True)

def spin1_operators():
    sq2 = np.sqrt(2.0)
    Sz = np.diag([1., 0., -1.])
    Sp = np.array([[0,sq2,0],[0,0,sq2],[0,0,0]], dtype=float)
    Sm = Sp.T.copy()
    Sx = 0.5*(Sp+Sm)
    Sy = -0.5j*(Sp-Sm)
    Id = np.eye(3)
    return Sz, Sp, Sm, Sx, Sy, Id

Sz, Sp, Sm, Sx, Sy, Id = spin1_operators()
print("Spin-1 algebra: OK")

def two_site_aklt():
    d = 3
    SS = (np.einsum('ij,kl->ikjl', Sz, Sz)
        + 0.5*np.einsum('ij,kl->ikjl', Sp, Sm)
        + 0.5*np.einsum('ij,kl->ikjl', Sm, Sp))
    SS_mat = SS.reshape(d*d, d*d)
    H2_mat = SS_mat + (1.0/3.0)*SS_mat @ SS_mat
    return H2_mat.reshape(d, d, d, d)   # [s1, s2, s1', s2']

H2_4 = two_site_aklt()

def build_aklt_mpo(L):
    """
    MPO convention: W[a_left, a_right, s, s']
    Chain contraction: H = W[0,a1]*W[a1,a2]*...*W[an,0]
    where matrix multiplication is over the MPO bond indices.

    W matrix (D_W x D_W, each entry a d x d operator):
         col:  0    1..r   r+1
    row 0  : [ I    0      0  ]   <- 'done' state: pass identity
    row 1..r: [ B_k  0      0  ]   <- 'active' state: output B_k, done
    row r+1 : [ 0    A_k    I  ]   <- 'waiting' state: start with A_k or pass

    Left boundary:  select row r+1  (start in 'waiting' state)
    Right boundary: select col 0    (end in 'done' state)

    So W_L[s,s'] = W[r+1, :, s, s']  -> shape (1, D_W, d, d)
       W_R[s,s'] = W[:, 0, s, s']    -> shape (D_W, 1, d, d)

    Two-site check:
        sum_b W_L[0,b,s1,S1] * W_R[b,0,s2,S2]
      = sum_b W[r+1, b, s1, S1] * W[b, 0, s2, S2]
      = W[r+1,0]*W[0,0] + W[r+1,1..r]*W[1..r,0] + W[r+1,r+1]*W[r+1,0]
      = 0*I + A_k*B_k + I*0
      = sum_k A_k * B_k  ✓
    """
    d = 3
    H2_r = H2_4.transpose(0, 2, 1, 3)      # [s1,s1',s2,s2']
    H2_mat = H2_r.reshape(d*d, d*d)

    U, sv, Vt = np.linalg.svd(H2_mat, full_matrices=False)
    r = int(np.sum(sv > 1e-12))
    U, sv, Vt = U[:, :r], sv[:r], Vt[:r, :]
    sqsv = np.sqrt(sv)
    A = (U * sqsv[None, :]).T.reshape(r, d, d)   # A[k, s, s']
    B = (Vt * sqsv[:, None]).reshape(r, d, d)    # B[k, s, s']
    print(f"H2 SVD rank = {r}")

    err = np.max(np.abs(np.einsum('ksu,ktv->sutv', A, B) - H2_r))
    print(f"Operator decomposition error: {err:.2e}", "OK" if err < 1e-10 else "FAIL")

    D_W = r + 2   # states: 0='done', 1..r='active with B_k', r+1='waiting'
    print(f"MPO bond dimension D_W = {D_W}")

    # W[a_in, a_out, s, s']
    W = np.zeros((D_W, D_W, d, d))

    # Row 0 ('done'): stay done, output identity
    W[0, 0, :, :] = np.eye(d)

    # Row r+1 ('waiting'): output A_k to activate, or stay waiting with identity
    W[r+1, r+1, :, :] = np.eye(d)          # pass through if not yet started
    for k in range(r):
        W[r+1, k+1, :, :] = A[k]           # transition: waiting -> active_k

    # Rows 1..r ('active_k'): output B_k and transition to done
    for k in range(r):
        W[k+1, 0, :, :] = B[k]             # transition: active_k -> done

    # Boundary tensors
    # Left end: MPS starts in 'waiting' state -> take row r+1
    W_L = W[r+1:r+2, :, :, :]              # shape (1, D_W, d, d)
    # Right end: MPS must end in 'done' state -> take col 0
    W_R = W[:, 0:1, :, :]                  # shape (D_W, 1, d, d)

    # Verify: sum_b W_L[0,b,s1,S1] * W_R[b,0,s2,S2] should equal H2_r[s1,S1,s2,S2]
    H2_check = np.einsum('ibsS,bjtT->sStT', W_L, W_R)   # [s1,S1,s2,S2]
    err2 = np.max(np.abs(H2_check - H2_r))
    print(f"Two-site MPO check error: {err2:.2e}", "OK" if err2 < 1e-10 else "FAIL")

    mpo = [W_L] + [W] * (L - 2) + [W_R]
    shapes = [m.shape for m in mpo]
    print(f"MPO tensor shapes: {[shapes[0], shapes[1], '...', shapes[-2], shapes[-1]]}")
    return mpo, D_W

mpo, D_W = build_aklt_mpo(20)


Spin-1 algebra: OK
H2 SVD rank = 9
Operator decomposition error: 1.10e-15 OK
MPO bond dimension D_W = 11
Two-site MPO check error: 1.10e-15 OK
MPO tensor shapes: [(1, 11, 3, 3), (11, 11, 3, 3), '...', (11, 11, 3, 3), (11, 1, 3, 3)]


In [2]:
# ============================================================
# Cell 2: MPS class + DMRG class + main computation
# ============================================================
import numpy as np
import time
from scipy.sparse.linalg import LinearOperator, eigsh

# ---------- Rebuild operators and MPO (Cell 2 self-contained) ----------
def spin1_operators():
    sq2 = np.sqrt(2.0)
    Sz = np.diag([1., 0., -1.])
    Sp = np.array([[0,sq2,0],[0,0,sq2],[0,0,0]], dtype=float)
    Sm = Sp.T.copy()
    Sx = 0.5*(Sp+Sm); Sy = -0.5j*(Sp-Sm); Id = np.eye(3)
    return Sz, Sp, Sm, Sx, Sy, Id

Sz, Sp, Sm, Sx, Sy, Id = spin1_operators()

def two_site_aklt():
    d = 3
    SS = (np.einsum('ij,kl->ikjl', Sz, Sz)
        + 0.5*np.einsum('ij,kl->ikjl', Sp, Sm)
        + 0.5*np.einsum('ij,kl->ikjl', Sm, Sp))
    SS_mat = SS.reshape(d*d, d*d)
    H2_mat = SS_mat + (1.0/3.0)*SS_mat @ SS_mat
    return H2_mat.reshape(d, d, d, d)

def build_aklt_mpo(L):
    d = 3
    H2_r = two_site_aklt().transpose(0, 2, 1, 3)
    H2_mat = H2_r.reshape(d*d, d*d)
    U, sv, Vt = np.linalg.svd(H2_mat, full_matrices=False)
    r = int(np.sum(sv > 1e-12))
    U, sv, Vt = U[:,:r], sv[:r], Vt[:r,:]
    sqsv = np.sqrt(sv)
    A = (U * sqsv[None,:]).T.reshape(r, d, d)
    B = (Vt * sqsv[:,None]).reshape(r, d, d)
    print(f"H2 SVD rank = {r}")
    err = np.max(np.abs(np.einsum('ksu,ktv->sutv', A, B) - H2_r))
    print(f"Operator decomposition error: {err:.2e}", "OK" if err < 1e-10 else "FAIL")
    D_W = r + 2
    print(f"MPO bond dimension D_W = {D_W}")
    W = np.zeros((D_W, D_W, d, d))
    W[0, 0, :, :] = np.eye(d)
    W[r+1, r+1, :, :] = np.eye(d)
    for k in range(r):
        W[r+1, k+1, :, :] = A[k]
        W[k+1, 0,   :, :] = B[k]
    W_L = W[r+1:r+2, :, :, :]
    W_R = W[:, 0:1,  :, :]
    err2 = np.max(np.abs(np.einsum('ibsS,bjtT->sStT', W_L, W_R) - H2_r))
    print(f"Two-site MPO check error: {err2:.2e}", "OK" if err2 < 1e-10 else "FAIL")
    return [W_L] + [W] * (L - 2) + [W_R], D_W

# ============================================================
class MPS:
    """Open-boundary MPS."""
    def __init__(self, L, d, D_max):
        self.L, self.d, self.D_max = L, d, D_max
        dims = [min(D_max, d**min(i, L-i)) for i in range(L+1)]
        self.tensors = []
        for i in range(L):
            Dl, Dr = dims[i], dims[i+1]
            self.tensors.append(np.random.randn(Dl, d, Dr))
        self._right_canonicalize()

    def _right_canonicalize(self):
        for i in range(self.L - 1, 0, -1):
            M = self.tensors[i]
            Dl, d_, Dr = M.shape
            Q, R = np.linalg.qr(M.reshape(Dl, d_*Dr).T)
            k = Q.shape[1]
            self.tensors[i]   = Q.T.reshape(k, d_, Dr)
            self.tensors[i-1] = np.einsum('ijk,kl->ijl', self.tensors[i-1], R.T)

    def norm_sq(self):
        env = np.ones((1, 1))
        for i in range(self.L):
            M = self.tensors[i]
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
        return float(np.real(env[0, 0]))

# ============================================================
class DMRG:
    """
    Two-site DMRG with MPO convention W[a_in, a_out, s, s'].

    Environment shapes:
        envL[i] : (a, x, u)   a=mpo bond, x=ket mps bond, u=bra mps bond
        envR[i] : (a, z, v)   a=mpo bond, z=ket mps bond, v=bra mps bond

    Left environment update (absorb site i from the left):
        envL_new[B, z, v] = sum_{a,x,u} envL[a,x,u] * M[x,s,z] * W[a,B,s,S] * Mc[u,S,v]
        einsum: 'axu, xsz, aBsS, uSv -> Bzv'

    Right environment update (absorb site i from the right):
        envR_new[a, x, u] = sum_{B,z,v} envR[B,z,v] * M[x,s,z] * W[a,B,s,S] * Mc[u,S,v]
        einsum: 'Bzv, xsz, aBsS, uSv -> axu'

    Matvec (two-site effective Hamiltonian):
        theta[x,si,sj,z], EL[a,x,u], Wi[a,b,si,Si], Wj[b,B,sj,Sj], ER[B,z,v]
        step 1: T[u,b,Si,sj,z] = sum_{a,x} EL[a,x,u] * theta[x,si,sj,z] * Wi[a,b,si,Si]
                einsum: 'axu, xijz, abij -> ubijz'  -- use distinct labels:
                einsum: 'axu, xisz, abiS -> ubSsz'  -- careful with repeated phys idx
        Use explicit two-step with clear index names below.
    """
    def __init__(self, mps, mpo):
        self.mps = mps
        self.mpo = mpo
        self.L   = mps.L
        self.d   = mps.d
        self.envL = [None] * (self.L + 1)
        self.envR = [None] * (self.L + 1)
        self._init_environments()

    def _init_environments(self):
        self.envL[0]      = np.ones((1, 1, 1))
        self.envR[self.L] = np.ones((1, 1, 1))
        self._right_canonicalize_all()
        for i in range(self.L - 1, -1, -1):
            self.envR[i] = self._update_envR(self.envR[i+1], i)

    def _right_canonicalize_all(self):
        mps = self.mps
        for i in range(self.L - 1, 0, -1):
            M = mps.tensors[i]
            Dl, d_, Dr = M.shape
            Q, R = np.linalg.qr(M.reshape(Dl, d_*Dr).T)
            k = Q.shape[1]
            mps.tensors[i]   = Q.T.reshape(k, d_, Dr)
            mps.tensors[i-1] = np.einsum('ijk,kl->ijl', mps.tensors[i-1], R.T)

    def _update_envL(self, envL_prev, site):
        """
        envL_prev : (a, x, u)
        M         : (x, s, z)    ket
        W         : (a, B, s, S) W[a_in, a_out, ket_phys, bra_phys]
        Mc        : (u, S, v)    bra
        output    : (B, z, v)
        """
        M  = self.mps.tensors[site]
        W  = self.mpo[site]
        Mc = M.conj()
        return np.einsum('axu,xsz,aBsS,uSv->Bzv', envL_prev, M, W, Mc)

    def _update_envR(self, envR_next, site):
        """
        envR_next : (B, z, v)
        M         : (x, s, z)    ket
        W         : (a, B, s, S) W[a_in, a_out, ket_phys, bra_phys]
        Mc        : (u, S, v)    bra
        output    : (a, x, u)
        """
        M  = self.mps.tensors[site]
        W  = self.mpo[site]
        Mc = M.conj()
        return np.einsum('Bzv,xsz,aBsS,uSv->axu', envR_next, M, W, Mc)

    def _matvec(self, v, EL, ER, Wi, Wj, Dl, d_, Dr):
        """
        EL    : (a, x, u)
        ER    : (B, z, v)
        Wi    : (a, b, si, Si)
        Wj    : (b, B, sj, Sj)
        theta : (x, si, sj, z)  reshaped from v
        output: (u, Si, Sj, v)  reshaped to flat
        """
        t = v.reshape(Dl, d_, d_, Dr)
        # Step 1: absorb EL and Wi
        # EL[a,x,u], t[x,si,sj,z], Wi[a,b,si,Si] -> T[u,b,Si,sj,z]
        T = np.einsum('axu,xisz,abiS->ubSsz', EL, t, Wi)
        # Step 2: absorb Wj and ER
        # T[u,b,Si,sj,z], Wj[b,B,sj,Sj], ER[B,z,v] -> out[u,Si,Sj,v]
        out = np.einsum('ubSsz,bBsT,Bzv->uSTv', T, Wj, ER)
        return out.reshape(-1)

    def _optimize_two_site(self, site_i):
        site_j = site_i + 1
        EL = self.envL[site_i]
        ER = self.envR[site_j + 1]
        Wi = self.mpo[site_i]
        Wj = self.mpo[site_j]

        Mi = self.mps.tensors[site_i]
        Mj = self.mps.tensors[site_j]
        Dl, d_, Dm = Mi.shape
        _,   _,  Dr = Mj.shape
        D_max = self.mps.D_max

        theta0 = np.einsum('ijk,klm->ijlm', Mi, Mj).reshape(-1)
        dim = Dl * d_ * d_ * Dr

        def matvec_fn(v):
            return self._matvec(v, EL, ER, Wi, Wj, Dl, d_, Dr)

        try:
            op = LinearOperator((dim, dim), matvec=matvec_fn, dtype=float)
            E, vec = eigsh(op, k=1, which='SA', v0=theta0, tol=1e-10, maxiter=1000)
            E0, theta = float(E[0]), vec[:, 0]
        except Exception:
            H_dense = np.zeros((dim, dim))
            for k in range(dim):
                ek = np.zeros(dim); ek[k] = 1.0
                H_dense[:, k] = matvec_fn(ek)
            H_dense = 0.5 * (H_dense + H_dense.T)
            evals, evecs = np.linalg.eigh(H_dense)
            E0, theta = float(evals[0]), evecs[:, 0]

        theta_mat = theta.reshape(Dl * d_, d_ * Dr)
        U, s, Vt = np.linalg.svd(theta_mat, full_matrices=False)
        D_new = min(D_max, int(np.sum(s > 1e-12 * s[0])) if s[0] > 0 else 1)
        D_new = max(D_new, 1)
        U, s, Vt = U[:, :D_new], s[:D_new], Vt[:D_new, :]
        self.mps.tensors[site_i] = U.reshape(Dl, d_, D_new)
        self.mps.tensors[site_j] = (np.diag(s) @ Vt).reshape(D_new, d_, Dr)
        return E0

    def sweep(self):
        L = self.L
        E = 0.0
        # Left-to-right
        for i in range(L - 1):
            E = self._optimize_two_site(i)
            Mi = self.mps.tensors[i]
            Dl, d_, Dr = Mi.shape
            Q, R = np.linalg.qr(Mi.reshape(Dl * d_, Dr))
            self.mps.tensors[i]   = Q.reshape(Dl, d_, -1)
            self.mps.tensors[i+1] = np.einsum('ij,jkl->ikl', R, self.mps.tensors[i+1])
            self.envL[i+1] = self._update_envL(self.envL[i], i)
        # Right-to-left
        for i in range(L - 2, -1, -1):
            E = self._optimize_two_site(i)
            Mj = self.mps.tensors[i+1]
            Dl, d_, Dr = Mj.shape
            Q, R = np.linalg.qr(Mj.reshape(Dl, d_ * Dr).T)
            self.mps.tensors[i+1] = Q.T.reshape(-1, d_, Dr)
            self.mps.tensors[i]   = np.einsum('ijk,kl->ijl', self.mps.tensors[i], R.T)
            self.envR[i+1] = self._update_envR(self.envR[i+2], i+1)
        return E

    def run(self, n_sweeps=20, tol=1e-8):
        L = self.L
        print(f"\nDMRG  L={L}  D_max={self.mps.D_max}  D_W={self.mpo[1].shape[1]}")
        print(f" {'Sweep':>6}  {'Energy':>20}  {'dE':>12}  {'time(s)':>8}")
        print("-" * 56)
        E_prev = np.inf
        energies = []
        for sw in range(1, n_sweeps + 1):
            t0 = time.time()
            E  = self.sweep()
            dE = abs(E - E_prev)
            dt = time.time() - t0
            print(f" {sw:>6}  {E:>20.10f}  {dE:>12.2e}  {dt:>6.2f}s")
            energies.append(E)
            if dE < tol and sw >= 3:
                print("  [Converged]")
                break
            E_prev = E
        return E, energies

# ============================================================
# Measurement utilities
# ============================================================
def expectation(mps_obj, op, site):
    """<psi|op_site|psi>"""
    env = np.ones((1, 1))
    for i in range(mps_obj.L):
        M = mps_obj.tensors[i]
        if i == site:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0, 0]))

def correlator(mps_obj, op1, op2, i, j):
    """<psi|op1_i op2_j|psi> for i < j"""
    env = np.ones((1, 1))
    for k in range(mps_obj.L):
        M = mps_obj.tensors[k]
        if k == i:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op1, M.conj())
        elif k == j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op2, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0, 0]))

def entanglement_entropy(mps_obj, bond):
    """Von Neumann entropy at given bond (1 to L-1)."""
    tensors = [t.copy() for t in mps_obj.tensors]
    for i in range(bond):
        M = tensors[i]
        Dl, d_, Dr = M.shape
        Q, R = np.linalg.qr(M.reshape(Dl * d_, Dr))
        tensors[i]   = Q.reshape(Dl, d_, -1)
        tensors[i+1] = np.einsum('ij,jkl->ikl', R, tensors[i+1])
    M = tensors[bond]
    Dl, d_, Dr = M.shape
    _, s, _ = np.linalg.svd(M.reshape(Dl * d_, Dr), full_matrices=False)
    s = s[s > 1e-14]; s /= np.sqrt(np.sum(s**2))
    return float(-np.sum(s**2 * np.log(s**2)))

def string_order(mps_obj, i, j):
    """<Sz_i exp(i*pi*sum_{i<k<j} Sz_k) Sz_j>"""
    Sz_ = np.diag([1., 0., -1.])
    eipSz = np.diag(np.exp(1j * np.pi * np.array([1., 0., -1.])))
    env = np.ones((1, 1), dtype=complex)
    for k in range(mps_obj.L):
        M = mps_obj.tensors[k].astype(complex)
        if k == i or k == j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, Sz_.astype(complex), M.conj())
        elif i < k < j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, eipSz, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0, 0]))

# ============================================================
# Main computation
# ============================================================
L, d, D = 20, 3, 5
print("=" * 60)
print(f"AKLT Ground State  L={L}  d={d}  D={D}")
print("=" * 60)

mpo, D_W = build_aklt_mpo(L)

np.random.seed(42)
mps = MPS(L, d, D)
bond_dims = [mps.tensors[0].shape[0]] + [m.shape[2] for m in mps.tensors]
print(f"Bond dims : {bond_dims}")
print(f"Norm^2    : {mps.norm_sq():.8f}")

dmrg = DMRG(mps, mpo)
E0, energies = dmrg.run(n_sweeps=30, tol=1e-10)

E_exact = -2/3 * (L - 1)
print(f"\nE0       = {E0:.10f}")
print(f"E0/bond  = {E0/(L-1):.10f}  (exact: {-2/3:.10f})")
print(f"Exact E0 = {E_exact:.10f}  [-2/3*(L-1)]")
print(f"Error    = {abs(E0 - E_exact):.2e}")


AKLT Ground State  L=20  d=3  D=5
H2 SVD rank = 9
Operator decomposition error: 1.10e-15 OK
MPO bond dimension D_W = 11
Two-site MPO check error: 1.10e-15 OK
Bond dims : [1, 3, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 5, 3, 1]
Norm^2    : 4027730026147893739520.00000000

DMRG  L=20  D_max=5  D_W=11
  Sweep                Energy            dE   time(s)
--------------------------------------------------------
      1        -12.6666665039           inf    4.07s
      2        -12.6666666667      1.63e-07    2.32s
      3        -12.6666666667      7.11e-15    1.79s
  [Converged]

E0       = -12.6666666667
E0/bond  = -0.6666666667  (exact: -0.6666666667)
Exact E0 = -12.6666666667  [-2/3*(L-1)]
Error    = 0.00e+00


In [4]:
# ============================================================
# Cell 3: Generate all 11 figures
# ============================================================
import numpy as np
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
import matplotlib.gridspec as gridspec
from scipy.optimize import curve_fit
import os

os.makedirs("figureAKLT", exist_ok=True)
plt.rcParams.update({'font.size': 12, 'figure.dpi': 150})

# Operators (redefined for Cell 3 self-containment)
sq2 = np.sqrt(2.0)
Sz_ = np.diag([1., 0., -1.])
Sp_ = np.array([[0,sq2,0],[0,0,sq2],[0,0,0]], dtype=float)
Sm_ = Sp_.T.copy()
Sx_ = 0.5*(Sp_ + Sm_)
Id_ = np.eye(3)

# ============================================================
# Measurement utilities (redefined for self-containment)
# ============================================================
def _expect(mps_obj, op, site):
    env = np.ones((1,1))
    for i in range(mps_obj.L):
        M = mps_obj.tensors[i]
        if i == site:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0,0]))

def _corr(mps_obj, op1, op2, i, j):
    env = np.ones((1,1))
    for k in range(mps_obj.L):
        M = mps_obj.tensors[k]
        if k == i:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op1, M.conj())
        elif k == j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, op2, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0,0]))

def _entropy(mps_obj, bond):
    tensors = [t.copy() for t in mps_obj.tensors]
    for i in range(bond):
        M = tensors[i]
        Dl, d_, Dr = M.shape
        Q, R = np.linalg.qr(M.reshape(Dl*d_, Dr))
        tensors[i]   = Q.reshape(Dl, d_, -1)
        tensors[i+1] = np.einsum('ij,jkl->ikl', R, tensors[i+1])
    M = tensors[bond]
    Dl, d_, Dr = M.shape
    _, s, _ = np.linalg.svd(M.reshape(Dl*d_, Dr), full_matrices=False)
    s = s[s > 1e-14]; s /= np.sqrt(np.sum(s**2))
    return float(-np.sum(s**2 * np.log(s**2)))

def _schmidt(mps_obj, bond):
    tensors = [t.copy() for t in mps_obj.tensors]
    for i in range(bond):
        M = tensors[i]
        Dl, d_, Dr = M.shape
        Q, R = np.linalg.qr(M.reshape(Dl*d_, Dr))
        tensors[i]   = Q.reshape(Dl, d_, -1)
        tensors[i+1] = np.einsum('ij,jkl->ikl', R, tensors[i+1])
    M = tensors[bond]
    Dl, d_, Dr = M.shape
    _, s, _ = np.linalg.svd(M.reshape(Dl*d_, Dr), full_matrices=False)
    s = s[s > 1e-14]; s /= np.sqrt(np.sum(s**2))
    return s

def _string_order(mps_obj, i, j):
    eipSz = np.diag(np.exp(1j*np.pi*np.array([1.,0.,-1.])))
    env = np.ones((1,1), dtype=complex)
    for k in range(mps_obj.L):
        M = mps_obj.tensors[k].astype(complex)
        if k == i or k == j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, Sz_.astype(complex), M.conj())
        elif i < k < j:
            env = np.einsum('xy,xsz,st,ytw->zw', env, M, eipSz, M.conj())
        else:
            env = np.einsum('xy,xsz,ysw->zw', env, M, M.conj())
    return float(np.real(env[0,0]))

# ============================================================
# Pre-compute all observables
# ============================================================
print("Computing observables...")

# Local <Sz>
Sz_exp = [_expect(mps, Sz_, i) for i in range(L)]
print(f"  <Sz> range: [{min(Sz_exp):.4f}, {max(Sz_exp):.4f}]")

# Spin-spin correlation from center
ref = L // 2
corr_vals = [_corr(mps, Sz_, Sz_, ref, j) for j in range(ref+1, L)]
dist_arr  = np.arange(1, len(corr_vals)+1)
print(f"  Correlation computed from site {ref}")

# Entanglement entropy at every bond
EE = [_entropy(mps, b) for b in range(1, L)]
print(f"  Entanglement entropy: center = {EE[L//2-1]:.4f}  (exact ln2 = {np.log(2):.4f})")

# Schmidt spectrum at center bond
sc = _schmidt(mps, L//2)
print(f"  Schmidt values at center: {sc}")

# String order parameter
i0 = L // 4
SO_dist = list(range(2, L//2))
SO_vals  = [_string_order(mps, i0, i0+r) for r in SO_dist]
print(f"  String order (long range) ~ {SO_vals[-1]:.4f}  (exact -4/9 = {-4/9:.4f})")

# Correlation matrix
cm_sites = list(range(1, L-1))
N_cm = len(cm_sites)
CM = np.zeros((N_cm, N_cm))
for a, si in enumerate(cm_sites):
    for b, sj in enumerate(cm_sites):
        if si == sj:
            CM[a,b] = _expect(mps, Sz_@Sz_, si)
        elif si < sj:
            CM[a,b] = _corr(mps, Sz_, Sz_, si, sj)
        else:
            CM[a,b] = _corr(mps, Sz_, Sz_, sj, si)
print("  Correlation matrix done")

print("All observables computed.\n")

# ============================================================
# Figure 01: DMRG Energy Convergence
# ============================================================
fig, ax = plt.subplots(figsize=(7,4))
sweeps = np.arange(1, len(energies)+1)
ax.plot(sweeps, energies, 'o-', color='steelblue', ms=6, lw=1.5, label='DMRG energy')
ax.axhline(-2/3*(L-1), color='crimson', ls='--', lw=1.5,
           label=f'Exact = {-2/3*(L-1):.6f}')
ax.set_xlabel('Sweep number')
ax.set_ylabel('Ground state energy')
ax.set_title(f'DMRG Energy Convergence  (L={L}, D={D})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/01_energy_convergence.png')
plt.close(fig)
print("01 saved: energy_convergence")

# ============================================================
# Figure 02: Local Magnetization <Sz_i>
# ============================================================
fig, ax = plt.subplots(figsize=(7,4))
ax.bar(range(L), Sz_exp, color='royalblue', alpha=0.75, edgecolor='navy', lw=0.5)
ax.axhline(0, color='k', lw=0.8)
ax.set_xlabel('Site $i$')
ax.set_ylabel(r'$\langle S^z_i \rangle$')
ax.set_title(f'Local Magnetization  (L={L}, D={D})')
ax.set_xlim(-0.5, L-0.5)
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/02_local_Sz.png')
plt.close(fig)
print("02 saved: local_Sz")

# ============================================================
# Figure 03: Spin-Spin Correlation + Exponential Fit
# ============================================================
xi_exact = 1.0 / np.log(3)   # exact AKLT correlation length

# Fit |C(d)| = A * exp(-d / xi)
abs_corr = np.abs(corr_vals)
try:
    def exp_decay(x, A, xi):
        return A * np.exp(-x / xi)
    popt, _ = curve_fit(exp_decay, dist_arr, abs_corr, p0=[abs_corr[0], xi_exact])
    A_fit, xi_fit = popt
    fit_label = f'Fit: A={A_fit:.3f}, ξ={xi_fit:.3f}'
except Exception:
    A_fit, xi_fit = abs_corr[0], xi_exact
    fit_label = 'Fit failed'

fig, ax = plt.subplots(figsize=(7,4))
ax.semilogy(dist_arr, abs_corr, 'o', color='steelblue', ms=5,
            label=r'$|\langle S^z_{r_0} S^z_{r_0+d} \rangle|$')
ax.semilogy(dist_arr, A_fit*np.exp(-dist_arr/xi_fit), '-',
            color='darkorange', lw=1.8, label=fit_label)
ax.semilogy(dist_arr, abs_corr[0]*np.exp(-dist_arr/xi_exact), '--',
            color='crimson', lw=1.5, label=f'Exact ξ = 1/ln3 = {xi_exact:.4f}')
ax.set_xlabel('Distance $d$')
ax.set_ylabel(r'$|\langle S^z S^z \rangle|$')
ax.set_title(f'Spin-Spin Correlation Function  (ref site = {ref})')
ax.legend(fontsize=10)
ax.grid(True, which='both', alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/03_spin_correlation.png')
plt.close(fig)
print("03 saved: spin_correlation")

# ============================================================
# Figure 04: Entanglement Entropy Profile
# ============================================================
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(range(1, L), EE, 's-', color='darkorange', ms=5, lw=1.5,
        label='Von Neumann entropy')
ax.axhline(np.log(2), color='crimson', ls='--', lw=1.5,
           label=f'Exact (bulk) = ln2 = {np.log(2):.4f}')
ax.set_xlabel('Bond index')
ax.set_ylabel('Entanglement entropy $S$')
ax.set_title(f'Entanglement Entropy Profile  (L={L}, D={D})')
ax.set_xlim(0, L)
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/04_entanglement_entropy.png')
plt.close(fig)
print("04 saved: entanglement_entropy")

# ============================================================
# Figure 05: Schmidt Spectrum at Center Bond
# ============================================================
fig, ax = plt.subplots(figsize=(7,4))
idx = np.arange(1, len(sc)+1)
ax.bar(idx, sc**2, color='mediumpurple', alpha=0.8, edgecolor='indigo', lw=0.8,
       label=r'$\lambda_\alpha^2$')
ax.axhline(0.5, color='crimson', ls='--', lw=1.5,
           label='Exact: 0.5 each (two-fold)')
ax.set_xlabel('Schmidt index $\\alpha$')
ax.set_ylabel(r'Schmidt weight $\lambda_\alpha^2$')
ax.set_title(f'Schmidt Spectrum at Center Bond  (bond {L//2})')
ax.set_xticks(idx)
ax.legend()
ax.grid(True, axis='y', alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/05_schmidt_spectrum.png')
plt.close(fig)
print("05 saved: schmidt_spectrum")

# ============================================================
# Figure 06: String Order Parameter
# ============================================================
fig, ax = plt.subplots(figsize=(7,4))
ax.plot(SO_dist, SO_vals, 'D-', color='seagreen', ms=5, lw=1.5,
        label=r'MPS: $\mathcal{O}^z(i_0, i_0+r)$')
ax.axhline(-4/9, color='crimson', ls='--', lw=1.5,
           label=f'Exact = $-4/9$ = {-4/9:.4f}')
ax.set_xlabel('Distance $r$')
ax.set_ylabel('String order parameter')
ax.set_title(f'String Order Parameter  (ref site $i_0$ = {i0})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/06_string_order.png')
plt.close(fig)
print("06 saved: string_order")

# ============================================================
# Figure 07: Energy vs Bond Dimension  (L=12)
# ============================================================
print("Computing Fig 07: energy vs D (L=12)...")
L12 = 12
E_exact_12 = -2/3 * (L12 - 1)
D_list = [2, 3, 4, 5, 6]
E_vs_D = []

for D_ in D_list:
    m12, _ = build_aklt_mpo(L12)
    np.random.seed(0)
    mps12  = MPS(L12, 3, D_)
    dmrg12 = DMRG(mps12, m12)
    E_, _  = dmrg12.run(n_sweeps=20, tol=1e-10)
    E_vs_D.append(E_)
    print(f"  L=12  D={D_}  E={E_:.8f}  err={abs(E_-E_exact_12):.2e}")

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(D_list, E_vs_D, 'o-', color='navy', ms=7, lw=1.5, label='DMRG')
ax.axhline(E_exact_12, color='crimson', ls='--', lw=1.5,
           label=f'Exact = {E_exact_12:.4f}')
ax.set_xlabel('Bond dimension $D$')
ax.set_ylabel('Ground state energy')
ax.set_title(f'Energy vs Bond Dimension  (L={L12})')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/07_energy_vs_D.png')
plt.close(fig)
print("07 saved: energy_vs_D")

# ============================================================
# Figure 08: Finite-Size Scaling
# ============================================================
print("Computing Fig 08: finite-size scaling...")
L_list  = [8, 10, 12, 14, 16, 20]
EperL   = []

for L_ in L_list:
    mp_, _ = build_aklt_mpo(L_)
    np.random.seed(0)
    mps_   = MPS(L_, 3, 5)
    dmrg_  = DMRG(mps_, mp_)
    E_, _  = dmrg_.run(n_sweeps=20, tol=1e-10)
    EperL.append(E_ / L_)
    print(f"  L={L_:<3}  E/L={E_/L_:.8f}  (exact bulk = {-2/3*(L_-1)/L_:.8f})")

inv_L = 1.0 / np.array(L_list)
# Linear extrapolation to 1/L -> 0
p = np.polyfit(inv_L, EperL, 1)
x_fit = np.linspace(0, max(inv_L)*1.1, 200)

fig, ax = plt.subplots(figsize=(7,4))
ax.plot(inv_L, EperL, 'o', color='darkcyan', ms=8, zorder=3, label='DMRG')
ax.plot(x_fit, np.polyval(p, x_fit), '--', color='darkorange', lw=1.5,
        label=f'Linear fit → {p[1]:.4f}')
ax.axhline(-2/3, color='crimson', ls=':', lw=1.5,
           label=f'Bulk exact = $-2/3$ = {-2/3:.4f}')
ax.set_xlabel('$1/L$')
ax.set_ylabel('$E_0 / L$')
ax.set_title('Finite-Size Scaling of Ground State Energy')
ax.legend()
ax.grid(True, alpha=0.3)
fig.tight_layout()
fig.savefig('figureAKLT/08_finite_size_scaling.png')
plt.close(fig)
print("08 saved: finite_size_scaling")

# ============================================================
# Figure 09: Correlation Matrix Heatmap
# ============================================================
vmax = np.max(np.abs(CM))
fig, ax = plt.subplots(figsize=(7,6))
im = ax.imshow(CM, cmap='RdBu_r', aspect='auto', vmin=-vmax, vmax=vmax,
               origin='upper')
cbar = plt.colorbar(im, ax=ax)
cbar.set_label(r'$\langle S^z_i S^z_j \rangle$')
ax.set_title(r'Spin Correlation Matrix $\langle S^z_i S^z_j \rangle$')
ax.set_xlabel('Site $j$')
ax.set_ylabel('Site $i$')
# Tick labels: show actual site indices
tick_step = max(1, N_cm // 8)
tick_idx  = list(range(0, N_cm, tick_step))
ax.set_xticks(tick_idx)
ax.set_yticks(tick_idx)
ax.set_xticklabels([cm_sites[t] for t in tick_idx])
ax.set_yticklabels([cm_sites[t] for t in tick_idx])
fig.tight_layout()
fig.savefig('figureAKLT/09_correlation_matrix.png')
plt.close(fig)
print("09 saved: correlation_matrix")

# ============================================================
# Figure 10: Summary Panel (2x2)
# ============================================================
fig = plt.figure(figsize=(13, 9))
gs  = gridspec.GridSpec(2, 2, hspace=0.38, wspace=0.32)

# (a) Energy convergence
ax = fig.add_subplot(gs[0,0])
ax.plot(np.arange(1, len(energies)+1), energies, 'o-',
        color='steelblue', ms=4, lw=1.5)
ax.axhline(-2/3*(L-1), color='crimson', ls='--', lw=1.2, label='Exact')
ax.set_xlabel('Sweep'); ax.set_ylabel('Energy')
ax.set_title('(a) DMRG Convergence')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# (b) Entanglement entropy
ax = fig.add_subplot(gs[0,1])
ax.plot(range(1,L), EE, 's-', color='darkorange', ms=4, lw=1.5)
ax.axhline(np.log(2), color='crimson', ls='--', lw=1.2,
           label=f'ln 2 = {np.log(2):.3f}')
ax.set_xlabel('Bond'); ax.set_ylabel('Entropy $S$')
ax.set_title('(b) Entanglement Entropy')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

# (c) Spin correlation (log scale)
ax = fig.add_subplot(gs[1,0])
ax.semilogy(dist_arr, abs_corr, 'o', color='steelblue', ms=4)
ax.semilogy(dist_arr, abs_corr[0]*np.exp(-dist_arr/xi_exact), '--',
            color='crimson', lw=1.5, label=f'Exact ξ={xi_exact:.3f}')
ax.set_xlabel('Distance'); ax.set_ylabel(r'$|\langle S^z S^z\rangle|$')
ax.set_title('(c) Spin Correlation')
ax.legend(fontsize=9); ax.grid(True, which='both', alpha=0.3)

# (d) String order
ax = fig.add_subplot(gs[1,1])
ax.plot(SO_dist, SO_vals, 'D-', color='seagreen', ms=4, lw=1.5)
ax.axhline(-4/9, color='crimson', ls='--', lw=1.2,
           label=f'Exact $-4/9$={-4/9:.3f}')
ax.set_xlabel('Distance $r$'); ax.set_ylabel('String order')
ax.set_title('(d) String Order Parameter')
ax.legend(fontsize=9); ax.grid(True, alpha=0.3)

fig.suptitle(f'AKLT Model Ground State  (L={L}, D={D})', fontsize=14, y=1.01)
fig.savefig('figureAKLT/10_summary_panel.png', bbox_inches='tight')
plt.close(fig)
print("10 saved: summary_panel")

# ============================================================
# Figure 11: VBS Structure Schematic
# ============================================================
fig, ax = plt.subplots(figsize=(13, 3.2))
ax.set_xlim(-0.6, L-0.4)
ax.set_ylim(-0.85, 1.4)
ax.axis('off')

box_h, box_w = 0.7, 0.65
virt_y = -0.45   # y-position of virtual spin-1/2 dots

for i in range(L):
    # Blue box: physical spin-1 site
    rect = plt.Rectangle((i - box_w/2, 0.0), box_w, box_h,
                          facecolor='lightsteelblue', edgecolor='navy',
                          lw=1.5, zorder=2)
    ax.add_patch(rect)
    ax.text(i, box_h/2, r'$S\!=\!1$', ha='center', va='center',
            fontsize=7.5, color='navy', zorder=3)

    # Virtual spin-1/2 dots below each box
    if i > 0:
        ax.plot(i - 0.18, virt_y + 0.18, 'o',
                color='crimson', ms=7, zorder=4)
        ax.plot([i - 0.18, i - 0.18], [virt_y + 0.18, 0.0],
                '-', color='crimson', lw=1.2, zorder=3)
    if i < L - 1:
        ax.plot(i + 0.18, virt_y + 0.18, 'o',
                color='crimson', ms=7, zorder=4)
        ax.plot([i + 0.18, i + 0.18], [virt_y + 0.18, 0.0],
                '-', color='crimson', lw=1.2, zorder=3)

# Singlet bonds (double-headed arrows between virtual spins)
for i in range(L - 1):
    ax.annotate('',
                xy=(i+1 - 0.18, virt_y + 0.18),
                xytext=(i + 0.18, virt_y + 0.18),
                arrowprops=dict(arrowstyle='<->', color='darkred',
                                lw=1.8, mutation_scale=12),
                zorder=5)

# Labels
ax.text(L/2 - 0.5, virt_y - 0.28,
        'Valence bond (singlet pair)',
        ha='center', fontsize=10, color='darkred')
ax.text(-0.5, box_h/2,
        r'$\mathcal{P}_{S=1}$',
        ha='right', va='center', fontsize=9, color='navy')

ax.set_title('AKLT Valence Bond Solid (VBS) Structure\n'
             r'Each spin-1 = symmetrized pair of virtual spin-$\frac{1}{2}$, '
             r'neighbors form singlets',
             fontsize=11, pad=8)
fig.tight_layout()
fig.savefig('figureAKLT/11_vbs_schematic.png', bbox_inches='tight')
plt.close(fig)
print("11 saved: vbs_schematic")

# ============================================================
# Final summary
# ============================================================
print("\n" + "="*60)
print("All 11 figures saved to figureAKLT/")
print("="*60)
print(f"  E0            = {energies[-1]:.10f}")
print(f"  Exact E0      = {-2/3*(L-1):.10f}")
print(f"  Error         = {abs(energies[-1]+2/3*(L-1)):.2e}")
print(f"  Center S(EE)  = {EE[L//2-1]:.6f}  (exact ln2 = {np.log(2):.6f})")
print(f"  Schmidt vals  = {sc}")
print(f"  String order  = {SO_vals[-1]:.6f}  (exact -4/9 = {-4/9:.6f})")
print(f"  Correlation ξ = {xi_fit:.4f}  (exact 1/ln3 = {xi_exact:.4f})")


Computing observables...
  <Sz> range: [-0.2645, 0.5585]
  Correlation computed from site 10
  Entanglement entropy: center = 0.6931  (exact ln2 = 0.6931)
  Schmidt values at center: [0.70712452 0.70708905]
  String order (long range) ~ -0.4444  (exact -4/9 = -0.4444)
  Correlation matrix done
All observables computed.

01 saved: energy_convergence
02 saved: local_Sz
03 saved: spin_correlation
04 saved: entanglement_entropy
05 saved: schmidt_spectrum
06 saved: string_order
Computing Fig 07: energy vs D (L=12)...
H2 SVD rank = 9
Operator decomposition error: 1.10e-15 OK
MPO bond dimension D_W = 11
Two-site MPO check error: 1.10e-15 OK

DMRG  L=12  D_max=2  D_W=11
  Sweep                Energy            dE   time(s)
--------------------------------------------------------
      1         -7.3333296737           inf    0.61s
      2         -7.3333333333      3.66e-06    0.46s
      3         -7.3333333333      1.23e-13    0.43s
  [Converged]
  L=12  D=2  E=-7.33333333  err=6.22e-15
H2 S